# Juggernaut XIII Ragnarok on Google Colab GPU

## Configuration

In [ ]:
MODEL_URL = "https://civitai.com/api/download/models/1759168?type=Model&format=SafeTensor&size=full&fp=fp16"
MODEL_PATH = "/content/models/juggernaut_xiii_ragnarok.safetensors"
HIRES_UPSCALER_URL = "https://huggingface.co/gemasai/4x_NMKD-Siax_200k/resolve/main/4x_NMKD-Siax_200k.pth"
HIRES_UPSCALER_PATH = "/content/upscalers/4x_NMKD-Siax_200k.pth"
OUTPUT_DIR = "/content/drive/MyDrive/colab_outputs"
DEFAULT_PROMPT = "Type in your prompt here make sure to leave the quotation marks"
DEFAULT_NEGATIVE_PROMPT = ""
DEFAULT_FILENAME_PREFIX = DEFAULT_PROMPT
DEFAULT_WIDTH = 1024
DEFAULT_HEIGHT = 1024
DEFAULT_STEPS = 40
DEFAULT_CFG = 3
DEFAULT_USE_KARRAS_SIGMAS = True
DEFAULT_NUM_IMAGES = 4
ENABLE_HIRES = True
HIRES_UPSCALE = 1.5

## 1. Install Dependencies

In [ ]:
!pip -q install --upgrade pip
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install -U diffusers transformers accelerate safetensors sentencepiece huggingface_hub

## 2. Mount Google Drive for Outputs

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Download Model Files

In [ ]:
import os
import time
import urllib.request
from safetensors import safe_open

def download_file(url, destination_path, label, *, retries=3):
    headers = {"User-Agent": "Mozilla/5.0"}

    for attempt in range(1, retries + 1):
        if os.path.exists(destination_path):
            os.remove(destination_path)

        request = urllib.request.Request(url, headers=headers)

        try:
            with urllib.request.urlopen(request) as response, open(destination_path, "wb") as out_file:
                expected_size = response.headers.get("Content-Length")
                expected_size = int(expected_size) if expected_size else None
                bytes_written = 0

                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    out_file.write(chunk)
                    bytes_written += len(chunk)

            if expected_size is not None and bytes_written != expected_size:
                raise OSError(f"Expected {expected_size} bytes but downloaded {bytes_written} bytes")

            print(f"Downloaded {label} to:", destination_path)
            print("Size (GB):", round(os.path.getsize(destination_path) / (1024 ** 3), 2))
            return
        except Exception as error:
            if os.path.exists(destination_path):
                os.remove(destination_path)
            if attempt == retries:
                raise RuntimeError(f"Failed to download {label} after {retries} attempts") from error
            print(f"Retrying {label} download after attempt {attempt} failed: {error}")
            time.sleep(3)

def validate_safetensors_file(checkpoint_path):
    with safe_open(checkpoint_path, framework="pt", device="cpu") as checkpoint:
        checkpoint.keys()

def download_validated_model(url, destination_path, *, retries=3):
    for attempt in range(1, retries + 1):
        try:
            download_file(url, destination_path, "model checkpoint", retries=1)
            validate_safetensors_file(destination_path)
            print("Safetensors checkpoint validation passed")
            return
        except Exception as error:
            if os.path.exists(destination_path):
                os.remove(destination_path)
            if attempt == retries:
                raise RuntimeError(f"Downloaded checkpoint is still invalid after {retries} attempts") from error
            print(f"Checkpoint validation failed on attempt {attempt}: {error}")
            time.sleep(3)

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

download_validated_model(MODEL_URL, MODEL_PATH)

if ENABLE_HIRES:
    os.makedirs(os.path.dirname(HIRES_UPSCALER_PATH), exist_ok=True)
    if os.path.exists(HIRES_UPSCALER_PATH):
        print("HiRes upscaler already present:", HIRES_UPSCALER_PATH)
    else:
        download_file(HIRES_UPSCALER_URL, HIRES_UPSCALER_PATH, "HiRes upscaler")
else:
    print("HiRes upscaler download skipped because ENABLE_HIRES is False")

## 4. Load the SDXL Pipelines

In [ ]:
import gc
import ctypes
from IPython import get_ipython
import torch
from diffusers import DPMSolverMultistepScheduler, StableDiffusionXLPipeline

try:
    LIBC = ctypes.CDLL("libc.so.6")
except OSError:
    LIBC = None

def trim_process_memory():
    if LIBC is not None:
        try:
            LIBC.malloc_trim(0)
        except Exception:
            pass

def clear_cuda_memory(*names):
    for name in names:
        obj = globals().pop(name, None)
        if obj is None:
            continue
        if hasattr(obj, "maybe_free_model_hooks"):
            try:
                obj.maybe_free_model_hooks()
            except Exception:
                pass
        if hasattr(obj, "remove_all_hooks"):
            try:
                obj.remove_all_hooks()
            except Exception:
                pass
        if hasattr(obj, "close"):
            try:
                obj.close()
            except Exception:
                pass
        if hasattr(obj, "components"):
            for component_name in list(obj.components.keys()):
                if hasattr(obj, component_name):
                    try:
                        setattr(obj, component_name, None)
                    except Exception:
                        pass
        del obj

    ip = get_ipython()
    if ip is not None:
        for cache_name in ["_", "__", "___"]:
            ip.user_ns.pop(cache_name, None)
        if hasattr(ip, "displayhook") and hasattr(ip.displayhook, "exec_result"):
            ip.displayhook.exec_result = None
        if hasattr(ip, "user_ns"):
            out_cache = ip.user_ns.get("Out")
            if hasattr(out_cache, "clear"):
                out_cache.clear()

    gc.collect()
    trim_process_memory()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def ensure_cuda_runtime():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available in this runtime. Connect to a Colab GPU runtime before loading the model.")

def configure_scheduler(pipeline):
    pipeline.scheduler = DPMSolverMultistepScheduler.from_config(
        pipeline.scheduler.config,
        algorithm_type="sde-dpmsolver++",
        solver_order=2,
        solver_type="midpoint",
        lower_order_final=True,
        euler_at_final=False,
        use_karras_sigmas=DEFAULT_USE_KARRAS_SIGMAS,
    )
    return pipeline

def build_sdxl_pipeline(*, cpu_offload=False):
    pipeline = StableDiffusionXLPipeline.from_single_file(
        MODEL_PATH,
        torch_dtype=torch.float16,
        use_safetensors=True,
    )
    pipeline = configure_scheduler(pipeline)
    pipeline.enable_attention_slicing()
    pipeline.vae.enable_slicing()

    if cpu_offload:
        pipeline.vae.enable_tiling()
        pipeline.enable_model_cpu_offload()
    else:
        pipeline = pipeline.to("cuda")

    return pipeline

ensure_cuda_runtime()
clear_cuda_memory("pipe", "stage_pipe", "hires_upsampler")

if ENABLE_HIRES:
    print("CUDA runtime ready. HiRes mode will reuse one SDXL pipeline for the full batch.")
else:
    pipe = build_sdxl_pipeline()
    print("SDXL text-to-image pipeline loaded successfully")

## 5. Batch Generate and Save Images

In [ ]:
import gc
import os
import numpy as np
import re
from datetime import datetime
from IPython.display import display
from PIL import Image

def save_and_preview_image(image, out_path):
    image.save(out_path)
    print("Saved to:", out_path)
    display(image)

os.makedirs(OUTPUT_DIR, exist_ok=True)

filename_stem = re.sub(r"[^a-z0-9]+", "_", DEFAULT_FILENAME_PREFIX.lower()).strip("_")
if not filename_stem:
    filename_stem = "image"
filename_stem = filename_stem[:80]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
generated_count = 0

if ENABLE_HIRES:
    target_width = int(round(DEFAULT_WIDTH * HIRES_UPSCALE))
    target_height = int(round(DEFAULT_HEIGHT * HIRES_UPSCALE))

    import math
    import torch.nn as nn

    class ResidualDenseBlock_5C(nn.Module):
        def __init__(self, nf=64, gc=32):
            super().__init__()
            self.conv1 = nn.Sequential(nn.Conv2d(nf, gc, 3, 1, 1, bias=True), nn.LeakyReLU(negative_slope=0.2, inplace=True))
            self.conv2 = nn.Sequential(nn.Conv2d(nf + gc, gc, 3, 1, 1, bias=True), nn.LeakyReLU(negative_slope=0.2, inplace=True))
            self.conv3 = nn.Sequential(nn.Conv2d(nf + gc * 2, gc, 3, 1, 1, bias=True), nn.LeakyReLU(negative_slope=0.2, inplace=True))
            self.conv4 = nn.Sequential(nn.Conv2d(nf + gc * 3, gc, 3, 1, 1, bias=True), nn.LeakyReLU(negative_slope=0.2, inplace=True))
            self.conv5 = nn.Sequential(nn.Conv2d(nf + gc * 4, nf, 3, 1, 1, bias=True))

        def forward(self, x):
            x1 = self.conv1(x)
            x2 = self.conv2(torch.cat((x, x1), 1))
            x3 = self.conv3(torch.cat((x, x1, x2), 1))
            x4 = self.conv4(torch.cat((x, x1, x2, x3), 1))
            x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
            return x5 * 0.2 + x

    class RRDB(nn.Module):
        def __init__(self, nf, gc=32):
            super().__init__()
            self.RDB1 = ResidualDenseBlock_5C(nf, gc)
            self.RDB2 = ResidualDenseBlock_5C(nf, gc)
            self.RDB3 = ResidualDenseBlock_5C(nf, gc)

        def forward(self, x):
            out = self.RDB1(x)
            out = self.RDB2(out)
            out = self.RDB3(out)
            return out * 0.2 + x

    class ShortcutBlock(nn.Module):
        def __init__(self, submodule):
            super().__init__()
            self.sub = submodule

        def forward(self, x):
            return x + self.sub(x)

    class OldRRDBNet(nn.Module):
        def __init__(self, in_nc, out_nc, nf, nb, gc=32, scale=4):
            super().__init__()
            rrdb_blocks = [RRDB(nf, gc=gc) for _ in range(nb)]
            trunk = nn.Sequential(*rrdb_blocks, nn.Conv2d(nf, nf, 3, 1, 1, bias=True))
            upsample_layers = int(math.log(scale, 2))

            model_layers = [
                nn.Conv2d(in_nc, nf, 3, 1, 1, bias=True),
                ShortcutBlock(trunk),
            ]

            for _ in range(upsample_layers):
                model_layers.extend([
                    nn.Upsample(scale_factor=2, mode="nearest"),
                    nn.Conv2d(nf, nf, 3, 1, 1, bias=True),
                    nn.LeakyReLU(negative_slope=0.2, inplace=True),
                ])

            model_layers.extend([
                nn.Conv2d(nf, nf, 3, 1, 1, bias=True),
                nn.LeakyReLU(negative_slope=0.2, inplace=True),
                nn.Conv2d(nf, out_nc, 3, 1, 1, bias=True),
            ])

            self.model = nn.Sequential(*model_layers)

        def forward(self, x):
            return self.model(x)

    def load_old_esrgan_model():
        loadnet = torch.load(HIRES_UPSCALER_PATH, map_location="cpu")
        if isinstance(loadnet, dict):
            if "params_ema" in loadnet:
                state_dict = loadnet["params_ema"]
            elif "params" in loadnet:
                state_dict = loadnet["params"]
            elif "state_dict" in loadnet:
                state_dict = loadnet["state_dict"]
            elif "model" in loadnet and isinstance(loadnet["model"], dict):
                state_dict = loadnet["model"]
            else:
                state_dict = loadnet
        else:
            state_dict = loadnet

        old_esrgan_model = OldRRDBNet(in_nc=3, out_nc=3, nf=64, nb=23, gc=32, scale=4)
        old_esrgan_model.load_state_dict(state_dict, strict=True)
        old_esrgan_model = old_esrgan_model.to("cuda").half().eval()
        del loadnet, state_dict
        gc.collect()
        print("Loaded HiRes upscaler with old ESRGAN-format checkpoint")
        return old_esrgan_model

    def upscale_with_old_esrgan(image, upscaler_model, tile_size=256, tile_pad=16):
        image_array = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
        image_tensor = torch.from_numpy(image_array).permute(2, 0, 1).unsqueeze(0)
        _, _, height, width = image_tensor.shape
        scale = 4
        output_tensor = torch.zeros((1, 3, height * scale, width * scale), dtype=torch.float32)

        for top in range(0, height, tile_size):
            for left in range(0, width, tile_size):
                input_start_y = max(top - tile_pad, 0)
                input_end_y = min(top + tile_size + tile_pad, height)
                input_start_x = max(left - tile_pad, 0)
                input_end_x = min(left + tile_size + tile_pad, width)

                input_tile = image_tensor[:, :, input_start_y:input_end_y, input_start_x:input_end_x].to("cuda", dtype=torch.float16)

                with torch.no_grad():
                    output_tile = upscaler_model(input_tile)

                crop_top = (top - input_start_y) * scale
                crop_left = (left - input_start_x) * scale
                crop_bottom = crop_top + min(tile_size, height - top) * scale
                crop_right = crop_left + min(tile_size, width - left) * scale

                output_start_y = top * scale
                output_end_y = output_start_y + min(tile_size, height - top) * scale
                output_start_x = left * scale
                output_end_x = output_start_x + min(tile_size, width - left) * scale

                output_tensor[:, :, output_start_y:output_end_y, output_start_x:output_end_x] = output_tile[:, :, crop_top:crop_bottom, crop_left:crop_right].float().cpu()
                del input_tile, output_tile

        torch.cuda.empty_cache()

        upscaled_array = output_tensor.squeeze(0).clamp(0, 1).permute(1, 2, 0).numpy()
        upscaled_image = Image.fromarray((upscaled_array * 255.0).round().astype(np.uint8))
        del image_tensor, output_tensor, upscaled_array
        gc.collect()

        if upscaled_image.size != (target_width, target_height):
            upscaled_image = upscaled_image.resize((target_width, target_height), Image.LANCZOS)

        return upscaled_image

    if not os.path.exists(HIRES_UPSCALER_PATH):
        raise RuntimeError("HiRes upscaler file is missing. Re-run Step 4 to download it.")

    clear_cuda_memory("pipe", "stage_pipe", "hires_upsampler")
    stage_pipe = build_sdxl_pipeline()

    for index in range(1, DEFAULT_NUM_IMAGES + 1):

        base_image = stage_pipe(
            prompt=DEFAULT_PROMPT,
            negative_prompt=DEFAULT_NEGATIVE_PROMPT,
            num_inference_steps=DEFAULT_STEPS,
            guidance_scale=DEFAULT_CFG,
            width=DEFAULT_WIDTH,
            height=DEFAULT_HEIGHT,
            num_images_per_prompt=1,
        ).images[0]
        hires_upsampler = load_old_esrgan_model()
        final_image = upscale_with_old_esrgan(base_image, hires_upsampler)
        clear_cuda_memory("hires_upsampler", "base_image")

        out_path = f"{OUTPUT_DIR}/{filename_stem}_{timestamp}_{index:02d}_hires.png"
        save_and_preview_image(final_image, out_path)
        generated_count += 1
        clear_cuda_memory("final_image")
        gc.collect()
else:
    for index, final_image in enumerate(
        pipe(
            prompt=DEFAULT_PROMPT,
            negative_prompt=DEFAULT_NEGATIVE_PROMPT,
            num_inference_steps=DEFAULT_STEPS,
            guidance_scale=DEFAULT_CFG,
            width=DEFAULT_WIDTH,
            height=DEFAULT_HEIGHT,
            num_images_per_prompt=DEFAULT_NUM_IMAGES,
        ).images,
        start=1,
    ):
        out_path = f"{OUTPUT_DIR}/{filename_stem}_{timestamp}_{index:02d}.png"
        save_and_preview_image(final_image, out_path)
        generated_count += 1
        clear_cuda_memory("final_image")
        gc.collect()

print(f"Generated {generated_count} image(s)")

clear_cuda_memory("pipe", "stage_pipe", "hires_upsampler", "base_image", "final_image")